In [1]:
# ============================================================
# 01 — Verificação da versão do DuckDB
# ============================================================

import duckdb
import pandas as pd
from pathlib import Path

print("DuckDB:", duckdb.__version__)

DuckDB: 1.5.5


In [2]:
# ============================================================
# 02 — Setup de caminhos e levantamento dos CSVs
# ============================================================

from pathlib import Path

ROOT = Path(r"C:\Users\bbert_xwhb76k\analises-GIS")
PROJECT = ROOT / "01-monitoramento-vegetacao"
RESULTS = PROJECT / "results"
PARQUET_DIR = PROJECT / "data" / "processed" / "parquet"

PARQUET_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RESULTS.rglob("*.csv"))

print(f"CSV encontrados: {len(csv_files)}")
print()

for f in csv_files:
    print(f.relative_to(PROJECT))

CSV encontrados: 28

results\tables\anomalias_por_data.csv
results\tables\correlacao_2020_indices.csv
results\tables\correlacao_2020_spearman.csv
results\tables\correlacao_2021_indices.csv
results\tables\correlacao_2021_spearman.csv
results\tables\correlacao_2022_indices.csv
results\tables\correlacao_2022_spearman.csv
results\tables\correlacao_2023_indices.csv
results\tables\correlacao_2023_spearman.csv
results\tables\correlacao_2024_indices.csv
results\tables\correlacao_2024_spearman.csv
results\tables\correlacao_2025_indices.csv
results\tables\correlacao_2025_spearman.csv
results\tables\correlacao_indices.csv
results\tables\estatisticas_espaciais.csv
results\tables\estatisticas_espaciais_2020.csv
results\tables\estatisticas_espaciais_2021.csv
results\tables\estatisticas_espaciais_2022.csv
results\tables\estatisticas_espaciais_2023.csv
results\tables\estatisticas_espaciais_2024.csv
results\tables\estatisticas_espaciais_2025.csv
results\tables\hotspots_anomalia.csv
results\tables\sazon

In [3]:
# ============================================================
# 03 — Inspeção do schema dos CSVs principais
# ============================================================

import duckdb

conn = duckdb.connect()

principais = [
    "serie_anual_completa.csv",
    "serie_intraanual_completa.csv",
    "anomalias_por_data.csv",
    "hotspots_anomalia.csv",
    "sazonalidade.csv",
    "estatisticas_espaciais.csv",
]

for nome in principais:
    arquivo = RESULTS / "tables" / nome

    print("\n" + "=" * 80)
    print(nome)
    print("=" * 80)

    df_info = conn.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto('{arquivo.as_posix()}')
    """).df()

    print(df_info.to_string(index=False))

conn.close()


serie_anual_completa.csv
column_name column_type null  key default extra
     indice     VARCHAR  YES None    None  None
        ano      BIGINT  YES None    None  None
      media      DOUBLE  YES None    None  None
    mediana      DOUBLE  YES None    None  None
        std      DOUBLE  YES None    None  None
        min      DOUBLE  YES None    None  None
        max      DOUBLE  YES None    None  None
    n_datas      BIGINT  YES None    None  None

serie_intraanual_completa.csv
column_name column_type null  key default extra
       data        DATE  YES None    None  None
   data_str      BIGINT  YES None    None  None
        ano      BIGINT  YES None    None  None
        mes      BIGINT  YES None    None  None
        dia      BIGINT  YES None    None  None
     indice     VARCHAR  YES None    None  None
      media      DOUBLE  YES None    None  None
    mediana      DOUBLE  YES None    None  None
        std      DOUBLE  YES None    None  None
        p25      DOUBLE  YES No

In [4]:
# ============================================================
# 04 — Conversão dos CSVs principais para Parquet
# ============================================================

from pathlib import Path
import duckdb

TABLES = RESULTS / "tables"
PARQUET_DIR = PROJECT / "data" / "processed" / "parquet"

PARQUET_DIR.mkdir(parents=True, exist_ok=True)

conn = duckdb.connect()

# ---------------------------------------------------------
# 1. Série anual
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        indice,
        CAST(ano AS INTEGER) AS ano,
        media,
        mediana,
        std,
        min,
        max,
        CAST(n_datas AS INTEGER) AS n_datas
    FROM read_csv_auto('{(TABLES / "serie_anual_completa.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "serie_anual.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 2. Série intra-anual
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        data,
        CAST(ano AS INTEGER) AS ano,
        CAST(mes AS INTEGER) AS mes,
        CAST(dia AS INTEGER) AS dia,
        indice,
        media,
        mediana,
        std,
        p25,
        p75,
        min,
        max,
        CAST(n_pixels AS INTEGER) AS n_pixels
    FROM read_csv_auto('{(TABLES / "serie_intraanual_completa.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "serie_intraanual.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 3. Anomalias
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        indice,
        data,
        CAST(ano AS INTEGER) AS ano,
        CAST(mes AS INTEGER) AS mes,
        media,
        mediana,
        std,
        CAST(n_pixels AS INTEGER) AS n_pixels,
        media_sazonal,
        std_sazonal,
        CAST(n AS INTEGER) AS n,
        anomalia,
        z_score,
        classificacao
    FROM read_csv_auto('{(TABLES / "anomalias_por_data.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "anomalias.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 4. Hotspots
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        indice,
        tipo,
        CAST(id_cluster AS INTEGER) AS id_cluster,
        CAST(n_pixels AS INTEGER) AS n_pixels,
        area_ha,
        limiar_persistencia_pct,
        limiar_anomalia
    FROM read_csv_auto('{(TABLES / "hotspots_anomalia.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "hotspots.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 5. Sazonalidade
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        indice,
        CAST(mes AS INTEGER) AS mes,
        media,
        mediana,
        std
    FROM read_csv_auto('{(TABLES / "sazonalidade.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "sazonalidade.parquet").as_posix()}'
(FORMAT PARQUET);
""")


# ---------------------------------------------------------
# 6. Estatísticas espaciais
# ---------------------------------------------------------

conn.execute(f"""
COPY (
    SELECT
        "Índice" AS indice,
        CAST("N válidos" AS INTEGER) AS n_validos,
        CAST("N NaN" AS INTEGER) AS n_nan,
        "Mín" AS minimo,
        P25 AS p25,
        Mediana AS mediana,
        Média AS media,
        P75 AS p75,
        "Máx" AS maximo,
        "Desv Pad" AS desvio_padrao
    FROM read_csv_auto('{(TABLES / "estatisticas_espaciais.csv").as_posix()}')
)
TO '{(PARQUET_DIR / "estatisticas_espaciais.parquet").as_posix()}'
(FORMAT PARQUET);
""")

conn.close()
print("Conversão concluída.")

Conversão concluída.


In [5]:
# ============================================================
# 05 — Verificação dos Parquets gerados
# ============================================================

parquets = sorted(PARQUET_DIR.glob("*.parquet"))

print(f"Parquets disponíveis: {len(parquets)}\n")

con = duckdb.connect()

for arquivo in parquets:
    n = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{arquivo.as_posix()}')"
    ).fetchone()[0]
    print(f"{arquivo.name:<35} {n:>8} linhas")

con.close()

Parquets disponíveis: 7

anomalias.parquet                        207 linhas
dim_data.parquet                        2192 linhas
estatisticas_espaciais.parquet             3 linhas
hotspots.parquet                          35 linhas
sazonalidade.parquet                      36 linhas
serie_anual.parquet                       18 linhas
serie_intraanual.parquet                 207 linhas


In [6]:
# ============================================================
# 05.1 — Validação de integridade e nulos por Parquet
# ============================================================

import duckdb

TABLES      = RESULTS / "tables"
PARQUET_DIR = PROJECT / "data" / "processed" / "parquet"

# Cada entrada descreve:
#   parquet      → arquivo de saída
#   csv          → arquivo de origem
#   colunas      → colunas-chave para checar nulos
validacoes = {
    "serie_anual.parquet": {
        "csv":     "serie_anual_completa.csv",
        "colunas": ["indice", "ano", "media"],
    },
    "serie_intraanual.parquet": {
        "csv":     "serie_intraanual_completa.csv",
        "colunas": ["data", "indice", "media"],
    },
    "anomalias.parquet": {
        "csv":     "anomalias_por_data.csv",
        "colunas": ["data", "indice", "z_score", "classificacao"],
    },
    "hotspots.parquet": {
        "csv":     "hotspots_anomalia.csv",
        "colunas": ["indice", "tipo", "area_ha"],
    },
    "sazonalidade.parquet": {
        "csv":     "sazonalidade.csv",
        "colunas": ["indice", "mes", "media"],
    },
    "estatisticas_espaciais.parquet": {
        "csv":     "estatisticas_espaciais.csv",
        "colunas": ["indice", "media"],
    },
}

conn = duckdb.connect()

# Cabeçalho
print(f"{'parquet':<35}{'csv':>8}{'pq':>8}{'status':>14}{'nulos':>10}")
print("-" * 75)

for parquet, cfg in validacoes.items():
    csv    = cfg["csv"]
    colunas = cfg["colunas"]

    # Contagem no CSV de origem
    n_csv = conn.execute(
        f"SELECT COUNT(*) FROM read_csv_auto('{(TABLES / csv).as_posix()}')"
    ).fetchone()[0]

    # Contagem no Parquet
    n_pq = conn.execute(
        f"SELECT COUNT(*) FROM read_parquet('{(PARQUET_DIR / parquet).as_posix()}')"
    ).fetchone()[0]

    # Soma de nulos em todas as colunas-chave
    selects = ", ".join(
        f'SUM(CASE WHEN "{c}" IS NULL THEN 1 ELSE 0 END)'
        for c in colunas
    )
    nulos = conn.execute(f"""
        SELECT {selects}
        FROM read_parquet('{(PARQUET_DIR / parquet).as_posix()}')
    """).fetchone()

    total_nulos = sum(nulos)

    status = "OK" if n_csv == n_pq else "DIVERGÊNCIA"
    flag   = "OK" if total_nulos == 0 else f"{total_nulos} NULOS"

    print(f"{parquet:<35}{n_csv:>8}{n_pq:>8}{status:>14}{flag:>10}")

conn.close()

parquet                                 csv      pq        status     nulos
---------------------------------------------------------------------------
serie_anual.parquet                      18      18            OK        OK
serie_intraanual.parquet                207     207            OK        OK
anomalias.parquet                       207     207            OK        OK
hotspots.parquet                         35      35            OK        OK
sazonalidade.parquet                     36      36            OK        OK
estatisticas_espaciais.parquet            3       3            OK        OK


In [7]:
# ============================================================
# 06 — Inspeção do schema dos Parquets
# ============================================================

import duckdb

conn = duckdb.connect()

for arquivo in sorted(PARQUET_DIR.glob("*.parquet")):
    print("\n" + "=" * 80)
    print(arquivo.name)
    print("=" * 80)

    resultado = conn.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{arquivo.as_posix()}')
    """).df()

    print(resultado.to_string(index=False))

conn.close()


anomalias.parquet
  column_name column_type null  key default extra
       indice     VARCHAR  YES None    None  None
         data        DATE  YES None    None  None
          ano     INTEGER  YES None    None  None
          mes     INTEGER  YES None    None  None
        media      DOUBLE  YES None    None  None
      mediana      DOUBLE  YES None    None  None
          std      DOUBLE  YES None    None  None
     n_pixels     INTEGER  YES None    None  None
media_sazonal      DOUBLE  YES None    None  None
  std_sazonal      DOUBLE  YES None    None  None
            n     INTEGER  YES None    None  None
     anomalia      DOUBLE  YES None    None  None
      z_score      DOUBLE  YES None    None  None
classificacao     VARCHAR  YES None    None  None

dim_data.parquet
column_name column_type null  key default extra
       data   TIMESTAMP  YES None    None  None
        ano      BIGINT  YES None    None  None
        mes      BIGINT  YES None    None  None
   nome_mes     VARCH

In [8]:
# ============================================================
# 07 — Preview das tabelas analíticas
# ============================================================

import duckdb
from IPython.display import display

conn = duckdb.connect()

for arquivo in sorted(PARQUET_DIR.glob("*.parquet")):
    print("\n" + "=" * 80)
    print(arquivo.name)
    print("=" * 80)

    df = conn.execute(f"""
        SELECT *
        FROM read_parquet('{arquivo.as_posix()}')
        LIMIT 5
    """).df()

    display(df)

conn.close()


anomalias.parquet


,indice,data,ano,mes,media,mediana,std,n_pixels,media_sazonal,std_sazonal,n,anomalia,z_score,classificacao
0,NDVI,2020-01-15,2020,1,0.309165,0.263145,0.293134,31080,0.414228,0.079520,5,-0.105063,-1.321209,Abaixo
1,NDVI,2020-02-14,2020,2,0.503374,0.766316,0.402265,31080,0.486078,0.057024,5,0.017296,0.303312,Normal
2,NDVI,2020-03-25,2020,3,0.522443,0.820190,0.413787,31080,0.390431,0.083181,6,0.132012,1.587039,Muito acima
3,NDVI,2020-04-04,2020,4,0.461980,0.554771,0.380069,31080,0.337883,0.062709,6,0.124097,1.978935,Muito acima
4,NDVI,2020-05-19,2020,5,0.300030,0.224212,0.304312,31080,0.331926,0.027087,6,-0.031896,-1.177514,Abaixo



dim_data.parquet


,data,ano,mes,nome_mes,trimestre,dia,dia_ano
0,2020-01-01,2020,1,January,T1,1,1
1,2020-01-02,2020,1,January,T1,2,2
2,2020-01-03,2020,1,January,T1,3,3
3,2020-01-04,2020,1,January,T1,4,4
4,2020-01-05,2020,1,January,T1,5,5



estatisticas_espaciais.parquet


...

,indice,mes,media,mediana,std
0,EVI,1,0.301545,0.385798,0.066381
1,EVI,2,0.403120,0.487031,0.111616
2,EVI,3,0.328998,0.333388,0.100545
3,EVI,4,0.209198,0.147345,0.049417
4,EVI,5,0.208365,0.181235,0.046967



serie_anual.parquet


,indice,ano,media,mediana,std,min,max,n_datas
0,EVI,2020,0.283348,0.192562,0.200283,0.000000,1.440746,12
1,EVI,2021,0.231034,0.214265,0.189550,0.000000,1.755039,12
2,EVI,2022,0.323643,0.383929,0.184404,-1.133468,2.820700,12
3,EVI,2023,0.250805,0.226347,0.202757,0.000000,0.932064,11
4,EVI,2024,0.256408,0.263644,0.176956,0.000000,1.543158,10



serie_intraanual.parquet


,data,ano,mes,dia,indice,media,mediana,std,p25,p75,min,max,n_pixels
0,2020-01-15,2020,1,15,EVI,0.223009,0.196038,0.216389,0.0,0.396003,0.0,0.814994,31080
1,2020-02-14,2020,2,14,EVI,0.366175,0.450718,0.316177,0.0,0.626960,0.0,1.018524,31080
2,2020-03-25,2020,3,25,EVI,0.380476,0.496871,0.315150,0.0,0.636567,0.0,0.891410,31080
3,2020-04-04,2020,4,4,EVI,0.307858,0.379992,0.256715,0.0,0.536238,0.0,0.841613,31080
4,2020-05-19,2020,5,19,EVI,0.155751,0.124041,0.164454,0.0,0.249630,0.0,0.696395,31080


In [9]:
# ============================================================
# 07.1 — Sumário e verificação de tipos dos Parquets
# ============================================================

import duckdb
import pandas as pd
from IPython.display import display

PARQUET_DIR = PROJECT / "data" / "processed" / "parquet"

# Tipos esperados por Parquet (mesmo conteúdo do bloco 07.2 original)
tipos_esperados = {
    "serie_anual.parquet": {
        "indice": "VARCHAR",
        "ano":    "INTEGER",
        "media":  "DOUBLE",
    },
    "anomalias.parquet": {
        "data":          "DATE",
        "indice":        "VARCHAR",
        "z_score":       "DOUBLE",
        "classificacao": "VARCHAR",
    },
    "hotspots.parquet": {
        "indice":  "VARCHAR",
        "tipo":    "VARCHAR",
        "area_ha": "DOUBLE",
    },
}

# ------------------------------------------------------------
# 1) Sumário visual (linhas, colunas, tamanho)
# ------------------------------------------------------------

resumo = []

for arquivo in sorted(PARQUET_DIR.glob("*.parquet")):
    linhas = duckdb.sql(
        f"SELECT COUNT(*) FROM read_parquet('{arquivo.as_posix()}')"
    ).fetchone()[0]

    colunas = duckdb.sql(
        f"SELECT COUNT(*) FROM (DESCRIBE SELECT * FROM read_parquet('{arquivo.as_posix()}'))"
    ).fetchone()[0]

    resumo.append({
        "parquet":    arquivo.name,
        "linhas":     linhas,
        "colunas":    colunas,
        "tamanho_kb": round(arquivo.stat().st_size / 1024, 1),
    })

display(pd.DataFrame(resumo))

# ------------------------------------------------------------
# 2) Verificação de tipos esperados
# ------------------------------------------------------------

for parquet, esperado in tipos_esperados.items():
    schema = duckdb.sql(f"""
        DESCRIBE SELECT * FROM read_parquet('{(PARQUET_DIR / parquet).as_posix()}')
    """).df()

    schema_dict = dict(zip(schema["column_name"], schema["column_type"]))

    print(f"\n{parquet}")
    for col, tipo in esperado.items():
        real = schema_dict.get(col)
        status = "OK" if real == tipo else f"ERRO ({real})"
        print(f"  {col:<20}{tipo:<12}{status}")

,parquet,linhas,colunas,tamanho_kb
0,anomalias.parquet,207,14,10.5
1,dim_data.parquet,2192,7,13.2
2,estatisticas_espaciais.parquet,3,10,1.4
3,hotspots.parquet,35,7,1.8
4,sazonalidade.parquet,36,5,1.6
5,serie_anual.parquet,18,8,1.8
6,serie_intraanual.parquet,207,13,10.3



serie_anual.parquet
  indice              VARCHAR     OK
  ano                 INTEGER     OK
  media               DOUBLE      OK

anomalias.parquet
  data                DATE        OK
  indice              VARCHAR     OK
  z_score             DOUBLE      OK
  classificacao       VARCHAR     OK

hotspots.parquet
  indice              VARCHAR     OK
  tipo                VARCHAR     OK
  area_ha             DOUBLE      OK


In [10]:
# ============================================================
# 08 — Criação do módulo analytics/connection.py
# ============================================================

from pathlib import Path

PROJECT = Path(r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao")
ANALYTICS = PROJECT / "analytics"

ANALYTICS.mkdir(exist_ok=True)

connection_code = '''from pathlib import Path
import duckdb


PROJECT_ROOT = Path(__file__).resolve().parents[1]
PARQUET_DIR = PROJECT_ROOT / "data" / "processed" / "parquet"


def get_connection():
    """
    Cria uma conexão DuckDB para o projeto.
    """
    return duckdb.connect()


def parquet_path(filename):
    """
    Retorna o caminho de um arquivo Parquet.
    """
    path = PARQUET_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Arquivo Parquet não encontrado: {path}"
        )

    return path


def read_parquet(filename):
    """
    Carrega um arquivo Parquet como DataFrame.
    """
    con = get_connection()
    path = parquet_path(filename)

    try:
        return con.execute(
            "SELECT * FROM read_parquet(?)",
            [str(path)]
        ).fetchdf()
    finally:
        con.close()
'''

(ANALYTICS / "connection.py").write_text(
    connection_code,
    encoding="utf-8"
)

print(f"Criado: {ANALYTICS / 'connection.py'}")

Criado: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\analytics\connection.py


In [11]:
# ============================================================
# 09 — Import do pacote analytics
# ============================================================

import sys

PROJECT = r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao"

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from analytics.connection import get_connection, parquet_path, read_parquet

print("Analytics carregado com sucesso.")

Analytics carregado com sucesso.


In [12]:
# ============================================================
# 10 — Versão do DuckDB via get_connection
# ============================================================

con = get_connection()

print("DuckDB:", con.execute("SELECT version()").fetchone()[0])

con.close()

DuckDB: v1.5.5


In [13]:
# ============================================================
# 11 — Teste de leitura de Parquet
# ============================================================

path = parquet_path("serie_anual.parquet")

print(path)
print(path.exists())

C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\parquet\serie_anual.parquet
True


In [14]:
# ============================================================
# 12 — Leitura da série anual via read_parquet
# ============================================================

df = read_parquet("serie_anual.parquet")

print("Dimensões:", df.shape)
display(df)

Dimensões: (18, 8)


,indice,ano,media,mediana,std,min,max,n_datas
0,EVI,2020,0.283348,0.192562,0.200283,0.000000,1.440746,12
1,EVI,2021,0.231034,0.214265,0.189550,0.000000,1.755039,12
2,EVI,2022,0.323643,0.383929,0.184404,-1.133468,2.820700,12
3,EVI,2023,0.250805,0.226347,0.202757,0.000000,0.932064,11
4,EVI,2024,0.256408,0.263644,0.176956,0.000000,1.543158,10
5,EVI,2025,0.239926,0.270757,0.195975,0.000000,0.959118,12
6,NDMI,2020,0.048193,0.000000,0.159472,-0.428050,0.551962,12
7,NDMI,2021,0.027494,0.000000,0.148518,-0.371102,0.549790,12
8,NDMI,2022,0.068558,0.000000,0.149714,-0.354516,0.572690,12
9,NDMI,2023,0.049757,0.000000,0.150537,-0.342539,0.524354,11


In [15]:
# ============================================================
# 13 — Criação do módulo analytics/queries.py
# ============================================================

from pathlib import Path

PROJECT = Path(r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao")
ANALYTICS = PROJECT / "analytics"

queries_code = '''from .connection import get_connection, parquet_path


def obter_serie_anual():
    """
    Retorna a série anual dos índices de vegetação.
    """
    con = get_connection()

    try:
        return con.execute("""
            SELECT *
            FROM read_parquet(?)
            ORDER BY ano, indice
        """, [str(parquet_path("serie_anual.parquet"))]).fetchdf()
    finally:
        con.close()


def obter_serie_intraanual():
    """
    Retorna a série temporal intraanual dos índices.
    """
    con = get_connection()

    try:
        return con.execute("""
            SELECT *
            FROM read_parquet(?)
            ORDER BY data, indice
        """, [str(parquet_path("serie_intraanual.parquet"))]).fetchdf()
    finally:
        con.close()


def obter_anomalias():
    """
    Retorna as anomalias temporais dos índices.
    """
    con = get_connection()

    try:
        return con.execute("""
            SELECT *
            FROM read_parquet(?)
            ORDER BY data, indice
        """, [str(parquet_path("anomalias.parquet"))]).fetchdf()
    finally:
        con.close()


def obter_hotspots():
    """
    Retorna os hotspots de anomalia.
    """
    con = get_connection()

    try:
        return con.execute("""
            SELECT *
            FROM read_parquet(?)
            ORDER BY indice, tipo, id_cluster
        """, [str(parquet_path("hotspots.parquet"))]).fetchdf()
    finally:
        con.close()


def obter_sazonalidade():
    """
    Retorna os valores sazonais históricos dos índices.
    """
    con = get_connection()

    try:
        return con.execute("""
            SELECT *
            FROM read_parquet(?)
            ORDER BY mes, indice
        """, [str(parquet_path("sazonalidade.parquet"))]).fetchdf()
    finally:
        con.close()


def obter_estatisticas_espaciais():
    """
    Retorna as estatísticas espaciais dos índices.
    """
    con = get_connection()

    try:
        return con.execute("""
            SELECT *
            FROM read_parquet(?)
            ORDER BY indice
        """, [str(parquet_path("estatisticas_espaciais.parquet"))]).fetchdf()
    finally:
        con.close()
'''

(ANALYTICS / "queries.py").write_text(
    queries_code,
    encoding="utf-8"
)

print(f"Criado: {ANALYTICS / 'queries.py'}")

Criado: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\analytics\queries.py


In [16]:
# ============================================================
# 14 — Import das funções de consulta
# ============================================================

from analytics.queries import (
    obter_serie_anual,
    obter_serie_intraanual,
    obter_anomalias,
    obter_hotspots,
    obter_sazonalidade,
    obter_estatisticas_espaciais
)

serie_anual = obter_serie_anual()

print("Serie anual:", serie_anual.shape)
display(serie_anual.head())

Serie anual: (18, 8)


,indice,ano,media,mediana,std,min,max,n_datas
0,EVI,2020,0.283348,0.192562,0.200283,0.000000,1.440746,12
1,NDMI,2020,0.048193,0.000000,0.159472,-0.428050,0.551962,12
2,NDVI,2020,0.360431,0.288021,0.325258,0.000000,0.953137,12
3,EVI,2021,0.231034,0.214265,0.189550,0.000000,1.755039,12
4,NDMI,2021,0.027494,0.000000,0.148518,-0.371102,0.549790,12


In [17]:
# ============================================================
# 15 — Carregamento das demais tabelas analíticas
# ============================================================

dados = {
    "serie_intraanual": obter_serie_intraanual(),
    "anomalias": obter_anomalias(),
    "hotspots": obter_hotspots(),
    "sazonalidade": obter_sazonalidade(),
    "estatisticas_espaciais": obter_estatisticas_espaciais(),
}

for nome, df in dados.items():
    print(f"{nome}: {df.shape}")

serie_intraanual: (207, 13)
anomalias: (207, 14)
hotspots: (35, 7)
sazonalidade: (36, 5)
estatisticas_espaciais: (3, 10)


In [18]:
# ============================================================
# 16 — Criação do módulo analytics/metrics.py
# ============================================================

from pathlib import Path

PROJECT = Path(r"C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao")
ANALYTICS = PROJECT / "analytics"

metrics_code = '''import pandas as pd

from .queries import (
    obter_serie_anual,
    obter_anomalias,
    obter_hotspots,
)


def obter_indicadores_atuais():
    """
    Retorna os principais indicadores do último período disponível
    para cada índice.
    """
    df = obter_anomalias()

    df["data"] = pd.to_datetime(df["data"])

    ultima_data = df["data"].max()

    atual = (
        df[df["data"] == ultima_data]
        .copy()
        .sort_values("indice")
    )

    return atual[
        [
            "indice",
            "data",
            "media",
            "media_sazonal",
            "anomalia",
            "z_score",
            "classificacao",
        ]
    ].reset_index(drop=True)


def calcular_diferenca_historica():
    """
    Calcula a diferença entre a média anual e a média histórica
    de cada índice.

    A média histórica é calculada sobre os anos disponíveis.
    """
    df = obter_serie_anual()

    historico = (
        df.groupby("indice", as_index=False)["media"]
        .mean()
        .rename(columns={"media": "media_historica"})
    )

    resultado = df.merge(
        historico,
        on="indice",
        how="left"
    )

    resultado["diferenca_historica"] = (
        resultado["media"] - resultado["media_historica"]
    )

    resultado["variacao_percentual"] = (
        resultado["diferenca_historica"]
        / resultado["media_historica"]
        * 100
    )

    return resultado


def obter_eventos_anomalos():
    """
    Conta os eventos classificados como diferentes do comportamento normal
    para cada índice.
    """
    df = obter_anomalias()

    eventos = (
        df.assign(
            evento_anomalo=~df["classificacao"].eq("Normal")
        )
        .groupby("indice", as_index=False)
        .agg(
            total_periodos=("data", "count"),
            eventos_anomalos=("evento_anomalo", "sum"),
        )
    )

    eventos["percentual_eventos_anomalos"] = (
        eventos["eventos_anomalos"]
        / eventos["total_periodos"]
        * 100
    )

    return eventos


def obter_area_hotspots():
    """
    Resume a área dos hotspots por índice e tipo de anomalia.
    """
    df = obter_hotspots()

    resultado = (
        df.groupby(
            ["indice", "tipo"],
            as_index=False
        )["area_ha"]
        .sum()
    )

    return resultado


def obter_resumo_hotspots():
    """
    Retorna a área positiva, negativa e total de hotspots
    para cada índice.
    """
    df = obter_hotspots()

    resumo = (
        df.pivot_table(
            index="indice",
            columns="tipo",
            values="area_ha",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
    )

    resumo.columns.name = None

    if "negativa" not in resumo.columns:
        resumo["negativa"] = 0

    if "positiva" not in resumo.columns:
        resumo["positiva"] = 0

    resumo["area_total_hotspots"] = (
        resumo["negativa"] + resumo["positiva"]
    )

    return resumo
'''

(ANALYTICS / "metrics.py").write_text(
    metrics_code,
    encoding="utf-8"
)

print(f"Criado: {ANALYTICS / 'metrics.py'}")

Criado: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\analytics\metrics.py


In [19]:
# ============================================================
# 17 — Indicadores atuais, comparação histórica e eventos
# ============================================================

from analytics.metrics import (
    obter_indicadores_atuais,
    calcular_diferenca_historica,
    obter_eventos_anomalos,
)

# Indicadores atuais
indicadores = obter_indicadores_atuais()

print("=== INDICADORES ATUAIS ===")
display(indicadores)


# Comparação histórica
historico = calcular_diferenca_historica()

print("=== COMPARAÇÃO HISTÓRICA ===")
display(historico)


# Eventos anômalos
eventos = obter_eventos_anomalos()

print("=== EVENTOS ANÔMALOS ===")
display(eventos)

=== INDICADORES ATUAIS ===


,indice,data,media,media_sazonal,anomalia,z_score,classificacao
0,EVI,2025-12-04,0.172766,0.354227,-0.181461,-0.878454,Abaixo
1,NDMI,2025-12-04,-0.029081,0.083935,-0.113016,-1.324028,Abaixo
2,NDVI,2025-12-04,0.255111,0.392747,-0.137636,-1.521022,Muito abaixo


=== COMPARAÇÃO HISTÓRICA ===


,indice,ano,media,mediana,std,min,max,n_datas,media_historica,diferenca_historica,variacao_percentual
0,EVI,2020,0.283348,0.192562,0.200283,0.000000,1.440746,12,0.264194,0.019154,7.249983
1,NDMI,2020,0.048193,0.000000,0.159472,-0.428050,0.551962,12,0.043030,0.005163,11.999384
2,NDVI,2020,0.360431,0.288021,0.325258,0.000000,0.953137,12,0.360560,-0.000129,-0.035819
3,EVI,2021,0.231034,0.214265,0.189550,0.000000,1.755039,12,0.264194,-0.033160,-12.551287
4,NDMI,2021,0.027494,0.000000,0.148518,-0.371102,0.549790,12,0.043030,-0.015536,-36.105175
5,NDVI,2021,0.340529,0.280599,0.310917,0.000000,0.949188,12,0.360560,-0.020031,-5.555507
6,EVI,2022,0.323643,0.383929,0.184404,-1.133468,2.820700,12,0.264194,0.059448,22.501801
7,NDMI,2022,0.068558,0.000000,0.149714,-0.354516,0.572690,12,0.043030,0.025528,59.326854
8,NDVI,2022,0.390908,0.461872,0.338342,0.000000,0.955195,12,0.360560,0.030348,8.416954
9,EVI,2023,0.250805,0.226347,0.202757,0.000000,0.932064,11,0.264194,-0.013389,-5.067827


=== EVENTOS ANÔMALOS ===


,indice,total_periodos,eventos_anomalos,percentual_eventos_anomalos
0,EVI,69,39,56.521739
1,NDMI,69,46,66.666667
2,NDVI,69,45,65.217391


In [20]:
# ============================================================
# 18 — Área e resumo consolidado dos hotspots
# ============================================================

from analytics.metrics import (
    obter_area_hotspots,
    obter_resumo_hotspots,
)

# Área dos hotspots
areas_hotspots = obter_area_hotspots()

print("=== ÁREA DOS HOTSPOTS ===")
display(areas_hotspots)


# Resumo dos hotspots
resumo_hotspots = obter_resumo_hotspots()

print("=== RESUMO DOS HOTSPOTS ===")
display(resumo_hotspots)

=== ÁREA DOS HOTSPOTS ===


,indice,tipo,area_ha
0,EVI,Negativo,11.92
1,NDMI,Negativo,40.28
2,NDMI,Positivo,1.24
3,NDVI,Negativo,46.97
4,NDVI,Positivo,15.08


=== RESUMO DOS HOTSPOTS ===


,indice,Negativo,Positivo,negativa,positiva,area_total_hotspots
0,EVI,11.92,0.00,0,0,0
1,NDMI,40.28,1.24,0,0,0
2,NDVI,46.97,15.08,0,0,0


In [21]:
# ============================================================
# 19 — Validação consolidada da camada Analytics
# ============================================================

print("=== VALIDAÇÃO DA CAMADA ANALYTICS ===\n")

print(f"Indicadores atuais:       {indicadores.shape}")
print(f"Comparação histórica:     {historico.shape}")
print(f"Eventos anômalos:         {eventos.shape}")
print(f"Área dos hotspots:        {areas_hotspots.shape}")
print(f"Resumo dos hotspots:      {resumo_hotspots.shape}")

print("\nÍndices encontrados:")

for indice in sorted(indicadores["indice"].unique()):
    print(f"  - {indice}")

print("\nÚltima data analisada:")
print(indicadores["data"].max())

print("\n✓ Analytics carregado e funcionando.")

=== VALIDAÇÃO DA CAMADA ANALYTICS ===

Indicadores atuais:       (3, 7)
Comparação histórica:     (18, 11)
Eventos anômalos:         (3, 4)
Área dos hotspots:        (5, 3)
Resumo dos hotspots:      (3, 6)

Índices encontrados:
  - EVI
  - NDMI
  - NDVI

Última data analisada:
2025-12-04 00:00:00

✓ Analytics carregado e funcionando.


In [22]:
# ============================================================
# 19.1 — Assertivas de sanidade
# ============================================================

import duckdb

PARQUET_DIR = PROJECT / "data" / "processed" / "parquet"

# 1. Toda tabela Parquet tem pelo menos 1 linha
for parquet in PARQUET_DIR.glob("*.parquet"):
    n = duckdb.sql(
        f"SELECT COUNT(*) FROM read_parquet('{parquet.as_posix()}')"
    ).fetchone()[0]
    assert n > 0, f"{parquet.name} está vazio"

# 2. Índices esperados
indices_esperados = {"NDVI", "EVI", "NDMI"}

indices_reais = set(
    duckdb.sql(
        f"SELECT DISTINCT indice FROM read_parquet('{(PARQUET_DIR / 'anomalias.parquet').as_posix()}')"
    ).df()["indice"]
)

assert indices_reais == indices_esperados, (
    f"Índices divergentes: {indices_reais} ≠ {indices_esperados}"
)

# 3. Última data dentro do período esperado
ultima = indicadores["data"].max()
assert str(ultima).startswith("2025"), f"Última data fora do esperado: {ultima}"

print("✓ Todas as assertivas passaram.")

✓ Todas as assertivas passaram.


In [23]:
# ============================================================
# 19.2 — Smoke tests das funções do pacote analytics
# ============================================================

from analytics.queries import (
    obter_serie_anual, obter_serie_intraanual, obter_anomalias,
    obter_hotspots, obter_sazonalidade, obter_estatisticas_espaciais,
)
from analytics.metrics import (
    obter_indicadores_atuais, calcular_diferenca_historica,
    obter_eventos_anomalos, obter_area_hotspots, obter_resumo_hotspots,
)

testes = {
    "obter_serie_anual":            (obter_serie_anual,            18,  8),
    "obter_serie_intraanual":       (obter_serie_intraanual,      207, 13),
    "obter_anomalias":              (obter_anomalias,             207, 14),
    "obter_hotspots":               (obter_hotspots,               35,  7),
    "obter_sazonalidade":           (obter_sazonalidade,           36,  5),
    "obter_estatisticas_espaciais": (obter_estatisticas_espaciais,  3, 10),
    "obter_indicadores_atuais":     (obter_indicadores_atuais,      3,  7),
    "obter_eventos_anomalos":       (obter_eventos_anomalos,        3,  4),
    "obter_area_hotspots":          (obter_area_hotspots,           5,  3),
    "obter_resumo_hotspots":        (obter_resumo_hotspots,         3,  6),
}

for nome, (funcao, n_linhas, n_colunas) in testes.items():
    df = funcao()
    assert df.shape == (n_linhas, n_colunas), (
        f"{nome}: esperado {(n_linhas, n_colunas)}, veio {df.shape}"
    )
    print(f"✓ {nome:<30} {df.shape}")

print("\n✓ Todos os smoke tests passaram.")

✓ obter_serie_anual              (18, 8)
✓ obter_serie_intraanual         (207, 13)
✓ obter_anomalias                (207, 14)
✓ obter_hotspots                 (35, 7)
✓ obter_sazonalidade             (36, 5)
✓ obter_estatisticas_espaciais   (3, 10)
✓ obter_indicadores_atuais       (3, 7)
✓ obter_eventos_anomalos         (3, 4)
✓ obter_area_hotspots            (5, 3)
✓ obter_resumo_hotspots          (3, 6)

✓ Todos os smoke tests passaram.


In [24]:
# ============================================================
# 20 — Criação da dimensão calendário (dim_data)
# ============================================================

from pathlib import Path
import duckdb

PROJECT_ROOT = Path(r"C:\Users\bbert_xwhb76k\analises-GIS")

PARQUET_DIR = (
    PROJECT_ROOT
    / "01-monitoramento-vegetacao"
    / "data"
    / "processed"
    / "parquet"
)

saida = PARQUET_DIR / "dim_data.parquet"

con = duckdb.connect()

con.execute("""
    COPY (
        SELECT
            data,
            YEAR(data) AS ano,
            MONTH(data) AS mes,
            strftime(data, '%B') AS nome_mes,
            'T' || CAST(QUARTER(data) AS VARCHAR) AS trimestre,
            DAY(data) AS dia,
            DAYOFYEAR(data) AS dia_ano
        FROM generate_series(
            DATE '2020-01-01',
            DATE '2025-12-31',
            INTERVAL 1 DAY
        ) AS t(data)
    )
    TO ?
    (FORMAT PARQUET, COMPRESSION ZSTD)
""", [str(saida)])

con.close()

print(f"Dimensão criada: {saida}")

Dimensão criada: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\parquet\dim_data.parquet


In [25]:
# ============================================================
# 21 — Leitura da dimensão calendário
# ============================================================

from analytics.connection import read_parquet

dim_data = read_parquet("dim_data.parquet")

print("=== DIMENSÃO DATA ===")
print(f"Registros:    {len(dim_data):,}")
print(f"Data inicial: {dim_data['data'].min()}")
print(f"Data final:   {dim_data['data'].max()}")
print(f"Colunas:      {list(dim_data.columns)}")

display(dim_data.head())

=== DIMENSÃO DATA ===
Registros:    2,192
Data inicial: 2020-01-01 00:00:00
Data final:   2025-12-31 00:00:00
Colunas:      ['data', 'ano', 'mes', 'nome_mes', 'trimestre', 'dia', 'dia_ano']


,data,ano,mes,nome_mes,trimestre,dia,dia_ano
0,2020-01-01,2020,1,January,T1,1,1
1,2020-01-02,2020,1,January,T1,2,2
2,2020-01-03,2020,1,January,T1,3,3
3,2020-01-04,2020,1,January,T1,4,4
4,2020-01-05,2020,1,January,T1,5,5


Arquivo criado:
C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\analytics\dashboard_data.py


In [30]:
# ============================================================
# 23 — Carregamento consolidado dos dados do dashboard
# ============================================================

import sys

PROJECT_ROOT = r"C:\Users\bbert_xwhb76k\analises-GIS"

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from analytics.dashboard_data import carregar_dados_dashboard

dados = carregar_dados_dashboard()

print("=== DADOS DO DASHBOARD ===\n")

for nome, df in dados.items():
    print(f"{nome:25} {df.shape}")

=== DADOS DO DASHBOARD ===

indicadores_atuais        (3, 7)
serie_anual               (18, 8)
serie_intraanual          (207, 13)
anomalias                 (207, 14)
sazonalidade              (36, 5)
hotspots                  (5, 3)
resumo_hotspots           (3, 6)
eventos_anomalos          (3, 4)
estatisticas_espaciais    (3, 10)
diferenca_historica       (18, 11)


In [31]:
# ============================================================
# 23.1 — Log de execução
# ============================================================

import json
import sys
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = r"C:\Users\bbert_xwhb76k\analises-GIS"

PARQUET_DIR = (
    Path(PROJECT_ROOT)
    / "01-monitoramento-vegetacao"
    / "data"
    / "processed"
    / "parquet"
)

log = {
    "executado_em":          datetime.now().isoformat(timespec="seconds"),
    "duckdb":                duckdb.__version__,
    "python":                sys.version.split()[0],
    "tabelas": {
        nome: list(df.shape)
        for nome, df in dados.items()
    },
    "ultima_data_analisada": str(indicadores["data"].max()),
    "indices":               sorted(indicadores["indice"].unique().tolist()),
}

saida_log = PARQUET_DIR / "_run_log.json"
saida_log.write_text(
    json.dumps(log, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Log salvo em: {saida_log}")
print(json.dumps(log, indent=2, ensure_ascii=False))

Log salvo em: C:\Users\bbert_xwhb76k\analises-GIS\01-monitoramento-vegetacao\data\processed\parquet\_run_log.json
{
  "executado_em": "2026-10-12T17:51:22",
  "duckdb": "1.5.5",
  "python": "3.11.16",
  "tabelas": {
    "indicadores_atuais": [
      3,
      7
    ],
    "serie_anual": [
      18,
      8
    ],
    "serie_intraanual": [
      207,
      13
    ],
    "anomalias": [
      207,
      14
    ],
    "sazonalidade": [
      36,
      5
    ],
    "hotspots": [
      5,
      3
    ],
    "resumo_hotspots": [
      3,
      6
    ],
    "eventos_anomalos": [
      3,
      4
    ],
    "estatisticas_espaciais": [
      3,
      10
    ],
    "diferenca_historica": [
      18,
      11
    ]
  },
  "ultima_data_analisada": "2025-12-04 00:00:00",
  "indices": [
    "EVI",
    "NDMI",
    "NDVI"
  ]
}
